# Climate Trend Analyzer - Exploratory Data Analysis

This reproducible notebook explores historical surface temperature anomalies provided by **NOAA NCEI (Climate at a Glance)** spanning 1850 to present.

### Objectives:
1. Load and validate empirical observation datasets.
2. Inspect summary statistics and data distributions.
3. Estimate long-term warming trends via OLS and Theil-Sen robust regression.
4. Detect statistical anomalies and unusual temperature departures.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is accessible
sys.path.insert(0, os.path.abspath(".."))

from src.data_loader import load_dataset
from src.data_cleaner import validate_schema, check_missing_values
from src.preprocessing import compute_rolling_averages, aggregate_annual
from src.trend_analysis import compute_ols_trend, compute_theil_sen_trend, compute_mann_kendall_test
from src.anomaly_detection import detect_anomalies

## 1. Data Ingestion & Quality Validation

In [ ]:
# Load Global (Land & Ocean) real observations
df, metadata = load_dataset("Global (Land & Ocean)")
print("Dataset Metadata:", metadata)
print(f"Observation count: {len(df):,}")
df.head()

In [ ]:
# Validate schema and inspect missing values
is_valid, issues = validate_schema(df)
missing_diag = check_missing_values(df)
print("Schema valid:", is_valid)
print("Missing values summary:", missing_diag)

## 2. Descriptive Summary Statistics

In [ ]:
df["Anomaly"].describe().round(4)

## 3. Long-Term Trend Estimation (Parametric & Non-Parametric)

In [ ]:
# Compute OLS and Theil-Sen regression
ols = compute_ols_trend(df)
theil = compute_theil_sen_trend(df)
mk = compute_mann_kendall_test(df)

print(f"OLS Decadal Warming Rate: +{ols['rate_per_decade']:.3f} °C/decade (95% CI: [{ols['ci_decade_lower']:.3f}, {ols['ci_decade_upper']:.3f}])")
print(f"Theil-Sen Robust Rate   : +{theil['theil_sen_rate_per_decade']:.3f} °C/decade")
print(f"Mann-Kendall Tau        : {mk['kendall_tau']:.3f} (Significant: {mk['is_significant']})")

## 4. Statistical Anomaly Detection

In [ ]:
# Detect statistical outliers using Z-score threshold
anom_results = detect_anomalies(df, method="zscore", z_threshold=2.5)
print(f"Total Flagged: {anom_results['flagged_count']} ({anom_results['flagged_percentage']} %)")
anom_results["flagged_table"][["Date", "Year", "Month", "Anomaly", "Z_Score", "Severity"]].head(10)